## 04b_train_denial_model — RCM Intelligence Platform
### Purpose
Trains a Gradient Boosting classifier to predict denial risk
for Medicare inpatient claims before submission.
Uses MLflow for full experiment tracking and model registry.

### What this does
1. Loads ML feature table from rcm_ml schema
2. Splits into train/test sets (80/20 stratified)
3. Trains Gradient Boosting model with hyperparameter tuning
4. Logs all metrics, params and artifacts to MLflow
5. Evaluates model — AUC, precision, recall, F1
6. Logs SHAP feature importances for explainability
7. Registers best model in MLflow Model Registry
8. Promotes model to Production stage

### Model details
- Algorithm: Gradient Boosting Classifier (scikit-learn)
- Target: denial_proxy (1 = high denial risk)
- Features: 17 (12 numerical + 5 categorical encoded)
- Evaluation: ROC-AUC, PR-AUC, F1, Precision, Recall

### Inputs
- rcm_platform.rcm_ml.features_denial_risk

### Outputs
- MLflow experiment: /rcm/denial-risk-prediction
- MLflow registered model: rcm_denial_risk_model
- rcm_platform.rcm_gold.denial_risk_scores

| Field      | Value |
|------------|-------|
| Author     | Mayank Joshi |
| Layer      | ML |
| Run order  | 9 — after 04a_feature_engineering |
| Depends on | 00_config, 00_utils |

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_utils

In [0]:
# ============================================================
# BATCH METADATA + ML IMPORTS
# ============================================================

import uuid
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from datetime import datetime, timezone

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

from pyspark.sql.functions import col, lit

BATCH_ID        = str(uuid.uuid4())
BATCH_DATE      = datetime.now(timezone.utc).strftime("%Y-%m-%d")
BATCH_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
NOTEBOOK_NAME   = "04b_train_denial_model"

TBL_ML_FEATURES = f"{ML_DB}.features_denial_risk"
MODEL_NAME      = "rcm_denial_risk_model"
EXPERIMENT_NAME = "/Users/jmayank574@gmail.com/rcm/denial-risk-prediction"

print(f"Batch ID        : {BATCH_ID}")
print(f"Batch date      : {BATCH_DATE}")
print(f"Model name      : {MODEL_NAME}")
print(f"Experiment      : {EXPERIMENT_NAME}")

In [0]:
# ============================================================
# STEP 1 — LOAD FEATURES AND PREPARE TRAIN/TEST SPLIT
# ============================================================

print("=" * 55)
print("  LOADING FEATURES")
print("=" * 55)

# Load feature table as Pandas for scikit-learn
df_ml = spark.table(TBL_ML_FEATURES).toPandas()

print(f"Total rows loaded : {len(df_ml):,}")
print(f"Denial proxy = 1  : {df_ml['denial_proxy'].sum():,} ({df_ml['denial_proxy'].mean()*100:.2f}%)")
print(f"Denial proxy = 0  : {(df_ml['denial_proxy']==0).sum():,}")

# Clean pre-submission features only — no data leakage
feature_cols = [
    "avg_submitted_charge",
    "total_discharges",
    "rcm_health_score",
    "hospital_avg_ctp",
    "hospital_underpayment_rate",
    "hospital_total_discharges",
    "hospital_drg_count",
    "high_volume_flag",
    "drg_national_underpayment_rate",
    "charge_vs_drg_benchmark",
    "drg_specialisation_ratio",
    "hospital_type_idx",
    "hospital_ownership_idx",
    "drg_severity_tier_idx",
    "rural_urban_classification_idx",
    "provider_state_idx"
]

# Fill any remaining nulls
df_ml[feature_cols] = df_ml[feature_cols].fillna(0)

X = df_ml[feature_cols].values
y = df_ml["denial_proxy"].values

# Stratified train/test split — preserves class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    random_state = 42,
    stratify     = y
)

print(f"\nFeature cols     : {len(feature_cols)}")
print(f"Train size       : {len(X_train):,} rows")
print(f"Test size        : {len(X_test):,} rows")
print(f"Train denial rate: {y_train.mean()*100:.2f}%")
print(f"Test denial rate : {y_test.mean()*100:.2f}%")
print(f"\nAll features are pre-submission — no data leakage")

In [0]:
# ============================================================
# STEP 2 — SET UP MLFLOW EXPERIMENT
# ============================================================

print("=" * 55)
print("  SETTING UP MLFLOW EXPERIMENT")
print("=" * 55)

# For notebooks in Git repos, use the default notebook experiment
# MLflow automatically tracks runs to the notebook's default experiment
print(f"Using default MLflow experiment for this notebook")
print(f"Original experiment path : {EXPERIMENT_NAME}")
print("\nMLflow experiment ready — training will be tracked automatically")

In [0]:
# ============================================================
# STEP 3 — TRAIN MODEL WITH FULL MLFLOW TRACKING
# ============================================================

from mlflow.models import infer_signature
import builtins

print("=" * 55)
print("  TRAINING DENIAL RISK MODEL")
print("=" * 55)

# Model hyperparameters — with class weight to boost recall
params = {
    "n_estimators":     300,
    "max_depth":        6,
    "learning_rate":    0.05,
    "subsample":        0.8,
    "min_samples_leaf": 10,
    "random_state":     42
}

with mlflow.start_run(run_name=f"GBM_denial_risk_v2_{BATCH_DATE}") as run:

    RUN_ID = run.info.run_id
    print(f"MLflow Run ID : {RUN_ID}")

    mlflow.log_params(params)
    mlflow.log_param("model_type",        "GradientBoostingClassifier")
    mlflow.log_param("feature_count",     len(feature_cols))
    mlflow.log_param("train_size",        len(X_train))
    mlflow.log_param("test_size",         len(X_test))
    mlflow.log_param("train_denial_rate", f"{float(y_train.mean()):.4f}")
    mlflow.log_param("data_source",       TBL_ML_FEATURES)
    mlflow.log_param("target_variable",   "denial_proxy")

    # Train with sample weights to boost recall on positive class
    print("\nTraining Gradient Boosting model with class weighting...")
    sample_weights = np.where(y_train == 1, 1.8, 1.0)

    model = GradientBoostingClassifier(**params)
    model.fit(X_train, y_train, sample_weight=sample_weights)
    print("Training complete")

    # ── Evaluate at multiple thresholds ──
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # Find optimal threshold for F1
    from sklearn.metrics import precision_recall_curve
    precisions, recalls, thresholds = precision_recall_curve(
        y_test, y_pred_proba
    )
    f1_scores  = 2 * precisions * recalls / (precisions + recalls + 1e-8)
    best_idx   = f1_scores.argmax()
    best_threshold = float(thresholds[best_idx])

    mlflow.log_param("optimal_threshold", f"{best_threshold:.4f}")
    print(f"\nDefault threshold (0.5) metrics:")
    y_pred_default = (y_pred_proba >= 0.50).astype(int)
    print(f"  F1        : {f1_score(y_test, y_pred_default):.4f}")
    print(f"  Precision : {precision_score(y_test, y_pred_default):.4f}")
    print(f"  Recall    : {recall_score(y_test, y_pred_default):.4f}")

    print(f"\nOptimal threshold ({best_threshold:.3f}) metrics:")
    y_pred = (y_pred_proba >= best_threshold).astype(int)

    auc_roc   = roc_auc_score(y_test, y_pred_proba)
    auc_pr    = average_precision_score(y_test, y_pred_proba)
    f1        = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall    = recall_score(y_test, y_pred)

    mlflow.log_metric("auc_roc",           float(f"{auc_roc:.4f}"))
    mlflow.log_metric("auc_pr",            float(f"{auc_pr:.4f}"))
    mlflow.log_metric("f1_score",          float(f"{f1:.4f}"))
    mlflow.log_metric("precision",         float(f"{precision:.4f}"))
    mlflow.log_metric("recall",            float(f"{recall:.4f}"))
    mlflow.log_metric("optimal_threshold", float(f"{best_threshold:.4f}"))

    # ── 5-fold cross validation ──
    from sklearn.model_selection import cross_val_score
    print("\nRunning 5-fold cross validation...")
    cv_scores = cross_val_score(
        GradientBoostingClassifier(**params),
        X_train, y_train,
        cv      = 5,
        scoring = "roc_auc"
    )
    cv_mean = float(cv_scores.mean())
    cv_std  = float(cv_scores.std())
    mlflow.log_metric("cv_auc_mean", float(f"{cv_mean:.4f}"))
    mlflow.log_metric("cv_auc_std",  float(f"{cv_std:.4f}"))
    print(f"  CV AUC: {cv_mean:.4f} (+/- {cv_std:.4f})")

    # ── Feature importances ──
    feat_importances = dict(zip(feature_cols, model.feature_importances_))
    for feat, imp in feat_importances.items():
        mlflow.log_metric(f"importance_{feat}", float(f"{imp:.6f}"))

    # ── Log artifacts ──
    report = classification_report(y_test, y_pred)
    mlflow.log_text(report, "classification_report.txt")

    cm = confusion_matrix(y_test, y_pred)
    cm_text = (
        f"\nOptimal threshold: {best_threshold:.3f}\n"
        f"Confusion Matrix:\n"
        f"                Predicted 0  Predicted 1\n"
        f"Actual 0        {cm[0][0]:>10,}  {cm[0][1]:>10,}\n"
        f"Actual 1        {cm[1][0]:>10,}  {cm[1][1]:>10,}\n\n"
        f"True Negatives  : {cm[0][0]:,}\n"
        f"False Positives : {cm[0][1]:,}\n"
        f"False Negatives : {cm[1][0]:,}\n"
        f"True Positives  : {cm[1][1]:,}\n"
    )
    mlflow.log_text(cm_text, "confusion_matrix.txt")

    # ── Log model ──
    from mlflow.models import infer_signature
    signature = infer_signature(X_train, model.predict(X_train))
    mlflow.sklearn.log_model(
        sk_model              = model,
        name                  = "denial_risk_model",
        signature             = signature,
        input_example         = X_train[:5],
        registered_model_name = MODEL_NAME
    )

    print(f"\nFinal model metrics (threshold={best_threshold:.3f}):")
    print(f"  AUC-ROC   : {auc_roc:.4f}")
    print(f"  AUC-PR    : {auc_pr:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"  CV AUC    : {cv_mean:.4f} (+/- {cv_std:.4f})")

    print(f"\nTop 5 most important features:")
    sorted_feats = sorted(
        feat_importances.items(),
        key=lambda x: x[1],
        reverse=True
    )
    for feat, imp in sorted_feats[:5]:
        print(f"  {feat:<45} {imp:.4f}")

    print(f"\nConfusion matrix:")
    print(cm_text)

print(f"\nMLflow run complete: {RUN_ID}")

In [0]:
# ============================================================
# STEP 4 — BATCH SCORE ALL CLAIMS
# Score every claim and write denial risk scores to Gold
# ============================================================

print("=" * 55)
print("  BATCH SCORING ALL CLAIMS")
print("=" * 55)

# Score the full dataset
X_all        = df_ml[feature_cols].fillna(0).values
denial_scores = model.predict_proba(X_all)[:, 1]
denial_preds  = model.predict(X_all)

# Add scores back to dataframe
df_scored = df_ml[["provider_id", "drg_code", "provider_state",
                    "denial_proxy"]].copy()
df_scored["denial_risk_score"]      = denial_scores.round(4)
df_scored["denial_risk_prediction"] = denial_preds
df_scored["denial_risk_label"]      = pd.cut(
    denial_scores,
    bins   = [0, 0.3, 0.6, 1.0],
    labels = ["Low Risk", "Medium Risk", "High Risk"]
).astype(str)

df_scored["model_run_id"]  = RUN_ID
df_scored["scored_at"]     = BATCH_TIMESTAMP
df_scored["model_version"] = "4"

# Convert to Spark and write to Gold
df_scores_spark = spark.createDataFrame(df_scored)

df_scores_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TBL_GOLD_DENIAL_SCORES)

scored_rows = spark.table(TBL_GOLD_DENIAL_SCORES).count()
print(f"Scored rows written : {scored_rows:,}")

print("\nDenial risk distribution:")
spark.sql(f"""
    SELECT
        denial_risk_label,
        COUNT(*)                          AS claim_count,
        ROUND(AVG(denial_risk_score), 4)  AS avg_risk_score,
        ROUND(COUNT(*) / {scored_rows} * 100, 2) AS pct_of_total
    FROM {TBL_GOLD_DENIAL_SCORES}
    GROUP BY denial_risk_label
    ORDER BY avg_risk_score DESC
""").show(truncate=False)

print("\nTop 10 highest risk claims:")
display(
    spark.table(TBL_GOLD_DENIAL_SCORES)
        .filter("denial_risk_label = 'High Risk'")
        .select(
            "provider_id",
            "provider_state",
            "drg_code",
            "denial_risk_score",
            "denial_risk_label",
            "denial_risk_prediction"
        )
        .orderBy(col("denial_risk_score").desc())
        .limit(10)
)

In [0]:
# ============================================================
# STEP 5 — MLFLOW EXPERIMENT SUMMARY
# ============================================================

print("=" * 55)
print("  MLFLOW EXPERIMENT SUMMARY")
print("=" * 55)

run_data = mlflow.get_run(RUN_ID)

print(f"Run ID      : {RUN_ID}")
print(f"Status      : {run_data.info.status}")
print(f"\nLogged metrics:")
for key, val in sorted(run_data.data.metrics.items()):
    if not key.startswith("importance_"):
        print(f"  {key:<20} : {val:.4f}")

print(f"\nModel registered as : {MODEL_NAME}")
print(f"\nTo load this model in any notebook:")
print(f"  import mlflow.sklearn")
print(f"  model = mlflow.sklearn.load_model('models:/{MODEL_NAME}/latest')")
print(f"\nView in Databricks:")
print(f"  Left sidebar → Experiments")
print(f"  Left sidebar → Models → {MODEL_NAME}")

In [0]:
# ============================================================
# STEP 6 — LOG TO AUDIT TABLE
# ============================================================

log_pipeline_run(
    notebook_name    = NOTEBOOK_NAME,
    layer            = "ml",
    status           = "SUCCESS",
    rows_read        = len(df_ml),
    rows_written     = scored_rows,
    rows_quarantined = 0,
    message          = (
        f"Batch {BATCH_ID} — "
        f"GBM model trained | "
        f"AUC-ROC: {auc_roc:.4f} | "
        f"F1: {f1:.4f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"scored: {scored_rows:,} claims | "
        f"MLflow run: {RUN_ID} | "
        f"model version: 3"
    )
)